# CNN Architecture Lab: AlexNet vs VGG16 vs Inception (GoogLeNet)

**Beginner-friendly Google Colab notebook**

In this notebook, we'll use a small subset of the online **CIFAR-10** dataset to compare three famous CNN architectures:

- **AlexNet** — an early deep CNN; uses ReLU and dropout.
- **VGG16** — stacks small 3×3 convolutions to build deeper feature extractors.
- **Inception / GoogLeNet** — uses parallel branches to capture features at different scales.

We'll use **transfer learning**: download ImageNet-pretrained models, freeze their feature-extraction layers, and train a new 10-class output layer for CIFAR-10. This makes the experiment much faster than training these large models from scratch.

> **Important:** This is a learning demo, not a fair benchmark. We use only a small dataset subset and a small number of epochs. Accuracy will vary.

## 1. Set up the environment

In Google Colab, select **Runtime → Change runtime type → T4 GPU** (or another GPU) if available. CPU mode can work, but it will be much slower.

The notebook uses PyTorch and TorchVision. Colab normally includes them already.

In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

## 2. Download a dataset from the internet

We'll use **CIFAR-10**, a public dataset containing 60,000 colour images in 10 categories. TorchVision downloads it automatically.

The original images are only 32×32 pixels. We'll resize them to 224×224 because these ImageNet-pretrained models are typically used with inputs around that size. We also apply ImageNet normalization, which matches the pretrained weights' expected input format.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 16

# Standard preprocessing for ImageNet-pretrained models
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_full = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
test_full = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# Use a small subset so the demo finishes sooner.
# Increase these numbers if you have more time and GPU memory.
NUM_TRAIN = 1000
NUM_TEST = 300

train_set = Subset(train_full, range(NUM_TRAIN))
test_set = Subset(test_full, range(NUM_TEST))

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    pin_memory=torch.cuda.is_available()
)

class_names = train_full.classes
print("Classes:", class_names)
print("Training images:", len(train_set))
print("Test images:", len(test_set))

### Look at a few dataset images

The images are normalized for the neural network, so we reverse that normalization just for display.

In [ ]:
images, labels = next(iter(train_loader))

def show_image(tensor_image, label):
    image = tensor_image.cpu().permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = np.clip(image * std + mean, 0, 1)
    plt.imshow(image)
    plt.title(class_names[label])
    plt.axis("off")

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    show_image(images[i], labels[i].item())
plt.tight_layout()
plt.show()

## 3. Understand the training idea

Each model already knows useful visual patterns from ImageNet. We'll:

1. Load its pretrained weights.
2. **Freeze** the existing layers so their weights do not change.
3. Replace the final classification layer with a new layer that outputs **10 scores** (one per CIFAR-10 class).
4. Train only that new final layer using cross-entropy loss and the Adam optimizer.

This is called **feature extraction transfer learning**. Later, you can unfreeze more layers to fine-tune the model.

In [ ]:
def freeze_all_parameters(model):
    for parameter in model.parameters():
        parameter.requires_grad = False

def prepare_alexnet():
    model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
    freeze_all_parameters(model)
    # AlexNet's final classifier layer predicts 1,000 ImageNet classes.
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)
    return model

def prepare_vgg16():
    model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    freeze_all_parameters(model)
    # Replace the final layer while keeping the pretrained feature extractor.
    model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)
    return model

def prepare_inception():
    # GoogLeNet is the original network built around Inception modules.
    model = models.googlenet(weights=models.GoogLeNet_Weights.DEFAULT)
    model.aux_logits = False
    freeze_all_parameters(model)
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model

model_builders = {
    "AlexNet": prepare_alexnet,
    "VGG16": prepare_vgg16,
    "Inception (GoogLeNet)": prepare_inception,
}

print("Model builders are ready.")

## 4. Check the architecture before training

We'll count total parameters and trainable parameters. Since the backbone is frozen, only the replacement classification layer should be trainable.

In [ ]:
demo_model = prepare_alexnet()
total_params = sum(p.numel() for p in demo_model.parameters())
trainable_params = sum(p.numel() for p in demo_model.parameters() if p.requires_grad)

print(demo_model)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

del demo_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Define the training and evaluation loops

**What happens in one training batch?**

- `model(images)` makes predictions (forward pass).
- `loss_fn(predictions, labels)` measures the mistakes.
- `loss.backward()` computes gradients.
- `optimizer.step()` updates the trainable weights.

During evaluation, we use `torch.no_grad()` because we don't need gradients.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    # Keep frozen feature extractors in evaluation mode.
    # This prevents BatchNorm running statistics from changing where applicable.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        # Some Inception implementations can return auxiliary outputs.
        if hasattr(outputs, "logits"):
            outputs = outputs.logits

        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)

            if hasattr(outputs, "logits"):
                outputs = outputs.logits

            loss = loss_fn(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

## 6. Train all three models

Start with **one epoch** to see the full workflow. You can increase `EPOCHS` to 2–5 after it works. The first run downloads pretrained weights, so it may take a little while.

The three models are trained independently on the same subset. The printed test accuracy is only a rough educational comparison.

In [ ]:
EPOCHS = 1
LEARNING_RATE = 1e-3

results = []
trained_models = {}

for model_name, build_model in model_builders.items():
    print("\n" + "=" * 65)
    print("Training:", model_name)

    start_time = time.time()
    model = build_model().to(device)

    # Only optimize parameters that we did not freeze.
    trainable_parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_parameters, lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, loss_fn, optimizer
        )
        test_loss, test_acc = evaluate(model, test_loader, loss_fn)

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"train loss: {train_loss:.4f} | "
            f"train accuracy: {train_acc:.2f}% | "
            f"test loss: {test_loss:.4f} | "
            f"test accuracy: {test_acc:.2f}%"
        )

    elapsed = time.time() - start_time
    results.append({
        "Model": model_name,
        "Train accuracy (%)": round(train_acc, 2),
        "Test accuracy (%)": round(test_acc, 2),
        "Time (seconds)": round(elapsed, 1)
    })
    trained_models[model_name] = model

    print(f"Time taken: {elapsed:.1f} seconds")

## 7. Compare the results

Don't be surprised if results are noisy. We trained only the final layer, used a small subset, and ran very few epochs. This is for learning the workflow—not for claiming that one architecture is universally best.

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values("Test accuracy (%)", ascending=False)
display(results_df)

plt.figure(figsize=(8, 4))
plt.bar(results_df["Model"], results_df["Test accuracy (%)"])
plt.ylabel("Test accuracy (%)")
plt.title("Small-subset CIFAR-10: educational comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8. Make a prediction with a trained model

Choose one of the models and see its prediction for a test image.

In [ ]:
chosen_model_name = "AlexNet"  # Change to "VGG16" or "Inception (GoogLeNet)"
chosen_model = trained_models[chosen_model_name]
chosen_model.eval()

image, true_label = test_full[0]

with torch.no_grad():
    logits = chosen_model(image.unsqueeze(0).to(device))
    if hasattr(logits, "logits"):
        logits = logits.logits
    probabilities = torch.softmax(logits, dim=1)[0]
    predicted_label = probabilities.argmax().item()

plt.figure(figsize=(4, 4))
show_image(image, true_label)
plt.title(
    f"True: {class_names[true_label]}\n"
    f"Predicted: {class_names[predicted_label]} "
    f"({probabilities[predicted_label].item() * 100:.1f}%)"
)
plt.show()

print("Top 3 predictions:")
top_probs, top_indices = torch.topk(probabilities, 3)
for prob, idx in zip(top_probs, top_indices):
    print(f"{class_names[idx.item()]}: {prob.item() * 100:.2f}%")

## 9. Save the trained classification heads

This saves each trained model's weights to the Colab session. To keep them after the session ends, download the files or save them to Google Drive.

In [ ]:
import os

os.makedirs("saved_models", exist_ok=True)

for model_name, model in trained_models.items():
    safe_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    path = f"saved_models/{safe_name}_cifar10_demo.pth"
    torch.save(model.state_dict(), path)
    print("Saved:", path)

## 10. Beginner recap

| Concept | What it means here |
|---|---|
| Dataset | CIFAR-10 images and their labels |
| Preprocessing | Resize and normalize images |
| Pretrained weights | Features learned earlier from ImageNet |
| Frozen layers | Existing weights stay unchanged |
| Classifier head | New layer that predicts 10 CIFAR-10 classes |
| Forward pass | Model produces class scores |
| Cross-entropy loss | Measures prediction error |
| Backpropagation | Computes gradients for trainable weights |
| Optimizer | Updates the trainable weights |
| Test accuracy | Fraction of test examples classified correctly |

### Try these experiments

1. Increase `NUM_TRAIN` to 3,000 and `NUM_TEST` to 1,000.
2. Change `EPOCHS` from 1 to 3.
3. Change `LEARNING_RATE` to `1e-4` and compare.
4. Try fine-tuning: unfreeze the final convolutional block and use a smaller learning rate.
5. Compare the model sizes and trainable parameter counts.

**Official references**
- [PyTorch CIFAR-10 training tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)
- [AlexNet source code](https://github.com/pytorch/vision/blob/main/torchvision/models/alexnet.py)
- [VGG source code](https://github.com/pytorch/vision/blob/main/torchvision/models/vgg.py)
- [GoogLeNet source code](https://github.com/pytorch/vision/blob/main/torchvision/models/googlenet.py)